# Experiment: Topic + Sentiment Analysis

Objective: discover latent topics and inspect how sentiment varies by dominant topic.


## 1. Setup


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.json"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.core import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[LABEL_COL].isin([0, 1]).all()
assert train_df[REVIEW_COL].str.len().gt(0).all()


## 5. Exploratory Analysis


In [ ]:
train_df["word_len"] = train_df[REVIEW_COL].str.split().str.len()
train_df["word_len"].describe(percentiles=[0.5, 0.9, 0.95])


## 6. Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(stop_words="english", min_df=5, max_features=10_000)
X_counts = vectorizer.fit_transform(train_df[REVIEW_COL])

lda = LatentDirichletAllocation(n_components=6, random_state=RANDOM_SEED)
X_topics = lda.fit_transform(X_counts)
train_df["topic_id"] = X_topics.argmax(axis=1)


## 7. Modeling


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

def top_words_for_topic(topic_idx: int, topn: int = 10):
    weights = lda.components_[topic_idx]
    top_idx = np.argsort(weights)[::-1][:topn]
    return ", ".join(feature_names[top_idx])

topic_terms = pd.DataFrame(
    {
        "topic_id": list(range(lda.n_components)),
        "top_terms": [top_words_for_topic(i, topn=10) for i in range(lda.n_components)],
    }
)
topic_terms


## 8. Evaluation


In [ ]:
topic_sentiment = pd.crosstab(train_df["topic_id"], train_df[LABEL_COL], normalize="index")
topic_sentiment = topic_sentiment.rename(columns={0: "neg_ratio", 1: "pos_ratio"}).reset_index()

merged = topic_sentiment.merge(topic_terms, on="topic_id", how="left")
merged


In [ ]:
ax = sns.barplot(data=merged, x="topic_id", y="pos_ratio")
ax.set_ylim(0, 1)
ax.set_title("Positive ratio by latent topic")
plt.tight_layout()
plt.show()


## 9. Inference / Submission


In [ ]:
out = PROCESSED_DIR / "topic_sentiment_summary.csv"
merged.to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT))
merged.head()


## 10. Conclusions / Next Steps

- Topic + sentiment analysis reveals which themes are associated with negative vs positive feedback.
- Topic labels are unsupervised and require manual interpretation.
- A next step is dynamic topic modeling or BERTopic for richer semantic grouping.
